# Доскональный EDA и ML-анализ для прогнозирования LTV пользователей Amazon

Цель блокнота: построить корректный аналитический пайплайн для задачи выявления поведенческих паттернов, влияющих на LTV пользователей онлайн-сервиса.

Логика анализа:
1. загрузка и первичная диагностика данных;
2. обработка типов, пропусков и аномалий;
3. формирование target `ltv` на будущий период;
4. построение пользовательских признаков на историческом периоде;
5. EDA: распределения, выбросы, корреляции, зависимости;
6. подготовка признаков к моделированию;
7. сравнение моделей;
8. интерпретация признаков через feature importance и permutation importance.


## 0. Импорт библиотек

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, cross_val_score, KFold
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.linear_model import Ridge, Lasso, LinearRegression
from sklearn.ensemble import RandomForestRegressor, HistGradientBoostingRegressor
from sklearn.inspection import permutation_importance

sns.set_theme(style='whitegrid')
pd.set_option('display.max_columns', 100)
pd.set_option('display.float_format', lambda x: f'{x:,.4f}')

## 1. Загрузка данных и первичная диагностика

Исходный датасет является транзакционным: одна строка соответствует одной покупке. Для прогнозирования LTV его нельзя напрямую подавать в модель, потому что объектом анализа является пользователь, а не отдельный заказ. Поэтому позже данные будут агрегированы на уровень пользователя.

In [ ]:
path = '/mnt/data/amazon-purchases.csv'
df_raw = pd.read_csv(path)
df_raw.head()

In [ ]:
df_raw.shape

In [ ]:
df_raw.info()

In [ ]:
df_raw.isna().sum().sort_values(ascending=False)

In [ ]:
df_raw.describe(include='all').T

## 2. Приведение названий колонок и типов данных

In [ ]:
df = df_raw.copy()

# Переименование колонок в удобный формат
df.columns = (
    df.columns
    .str.lower()
    .str.strip()
    .str.replace(' ', '_')
    .str.replace('/', '_')
    .str.replace('(', '', regex=False)
    .str.replace(')', '', regex=False)
)

# Для удобства переименуем ключевые поля
df = df.rename(columns={
    'asin_isbn_product_code': 'asin_isbn',
    'survey_responseid': 'user_id'
})

# Даты и денежный признак
df['order_date'] = pd.to_datetime(df['order_date'], errors='coerce')
df['total_price'] = df['purchase_price_per_unit'] * df['quantity']

# Пропуски в категориальных признаках не удаляем, а выделяем в отдельную категорию
df['shipping_address_state'] = df['shipping_address_state'].fillna('unknown')
df['title'] = df['title'].fillna('unknown')
df['asin_isbn'] = df['asin_isbn'].fillna('unknown')
df['category'] = df['category'].fillna('unknown')

# Удаляем только невозможные наблюдения, если они есть
df = df.dropna(subset=['order_date', 'user_id'])
df = df[(df['purchase_price_per_unit'] > 0) & (df['quantity'] > 0) & (df['total_price'] > 0)]

df.head()

In [ ]:
df.shape, df['user_id'].nunique(), df['category'].nunique(), df['order_date'].min(), df['order_date'].max()

## 3. Первичный EDA транзакционного уровня

На этом этапе анализируются сами транзакции: распределение цен, количества товаров, общей стоимости заказа, динамика покупок во времени и пропуски.

In [ ]:
missing = df_raw.isna().mean().sort_values(ascending=False).reset_index()
missing.columns = ['column', 'missing_share']
missing

In [ ]:
plt.figure(figsize=(10, 5))
sns.barplot(data=missing, x='missing_share', y='column')
plt.title('Доля пропусков по признакам')
plt.xlabel('Доля пропусков')
plt.ylabel('Признак')
plt.show()

In [ ]:
num_cols_transaction = ['purchase_price_per_unit', 'quantity', 'total_price']
df[num_cols_transaction].describe(percentiles=[.01, .05, .25, .5, .75, .95, .99]).T

In [ ]:
for col in num_cols_transaction:
    plt.figure(figsize=(9, 4))
    sns.histplot(np.log1p(df[col]), bins=60)
    plt.title(f'Лог-распределение {col}')
    plt.xlabel(f'log1p({col})')
    plt.show()

In [ ]:
monthly_orders = df.set_index('order_date').resample('M')['user_id'].count()
monthly_revenue = df.set_index('order_date').resample('M')['total_price'].sum()

plt.figure(figsize=(14, 4))
monthly_orders.plot()
plt.title('Количество транзакций по месяцам')
plt.ylabel('Количество транзакций')
plt.show()

plt.figure(figsize=(14, 4))
monthly_revenue.plot()
plt.title('Выручка по месяцам')
plt.ylabel('Выручка')
plt.show()

## 4. Корректная постановка задачи LTV

Чтобы избежать утечки данных, признаки считаются только на историческом периоде `past`, а целевая переменная `ltv` — на будущем периоде `future`.

В этом варианте используется дата отсечения `2021-01-01`. Её можно изменить, но важно, чтобы:
- `X` строился до даты отсечения;
- `y` строился после даты отсечения;
- модель не использовала информацию из будущего.

In [ ]:
cutoff = pd.Timestamp('2021-01-01')

past = df[df['order_date'] < cutoff].copy()
future = df[df['order_date'] >= cutoff].copy()

past.shape, future.shape, past['user_id'].nunique(), future['user_id'].nunique()

In [ ]:
ltv = (
    future.groupby('user_id')['total_price']
    .sum()
    .rename('ltv')
    .reset_index()
)

ltv.head()

## 5. Feature Engineering: формирование признаков на уровне пользователя

Именно этот этап является ключевым. Здесь формируются признаки, которые описывают не только объём покупок, но и поведенческий паттерн: частоту, регулярность, раннюю активность, разнообразие категорий и изменение поведения во времени.

In [ ]:
def category_entropy(x):
    counts = x.value_counts(normalize=True)
    return -(counts * np.log1p(counts)).sum()

# Базовая агрегация по пользователю
user = past.groupby('user_id').agg(
    total_revenue=('total_price', 'sum'),
    num_orders=('total_price', 'count'),
    total_quantity=('quantity', 'sum'),
    avg_order_value=('total_price', 'mean'),
    median_order_value=('total_price', 'median'),
    std_order_value=('total_price', 'std'),
    min_order_value=('total_price', 'min'),
    max_order_value=('total_price', 'max'),
    avg_unit_price=('purchase_price_per_unit', 'mean'),
    first_order_date=('order_date', 'min'),
    last_order_date=('order_date', 'max'),
    active_days=('order_date', lambda x: x.dt.date.nunique()),
    category_count=('category', 'nunique'),
    product_count=('asin_isbn', 'nunique'),
    state_count=('shipping_address_state', 'nunique'),
    category_entropy=('category', category_entropy)
).reset_index()

# Временные признаки
user['recency'] = (cutoff - user['last_order_date']).dt.days
user['customer_age'] = (cutoff - user['first_order_date']).dt.days
user['tenure'] = (user['last_order_date'] - user['first_order_date']).dt.days

# Защита от деления на ноль
user['customer_age_safe'] = user['customer_age'].replace(0, 1)
user['tenure_safe'] = user['tenure'].replace(0, 1)

# Интенсивность поведения
user['purchase_frequency_rate'] = user['num_orders'] / user['customer_age_safe']
user['revenue_per_day'] = user['total_revenue'] / user['customer_age_safe']
user['orders_per_active_day'] = user['num_orders'] / user['active_days'].replace(0, 1)
user['active_days_ratio'] = user['active_days'] / user['customer_age_safe']
user['average_order_size'] = user['total_quantity'] / user['num_orders'].replace(0, 1)
user['recency_normalized'] = user['recency'] / user['customer_age_safe']

# Коэффициент вариации: стабильность среднего чека
user['cv_order_value'] = user['std_order_value'] / user['avg_order_value'].replace(0, np.nan)

user.head()

### 5.1 Интервалы между покупками

Средний интервал показывает частоту, а стандартное отклонение интервалов — стабильность поведения. Для LTV это часто важнее, чем просто суммарная выручка.

In [ ]:
past_sorted = past.sort_values(['user_id', 'order_date']).copy()
past_sorted['days_between_orders'] = past_sorted.groupby('user_id')['order_date'].diff().dt.days

interval_features = past_sorted.groupby('user_id')['days_between_orders'].agg(
    avg_days_between_orders='mean',
    median_days_between_orders='median',
    std_days_between_orders='std',
    min_days_between_orders='min',
    max_days_between_orders='max'
).reset_index()

user = user.merge(interval_features, on='user_id', how='left')
user.head()

### 5.2 Раннее поведение пользователя

Признаки первых 30 дней после первой покупки помогают понять, насколько быстро пользователь начинает приносить ценность.

In [ ]:
first_dates = past.groupby('user_id')['order_date'].min().rename('first_date')
past_early = past.merge(first_dates, on='user_id', how='left')
past_early['days_from_start'] = (past_early['order_date'] - past_early['first_date']).dt.days

early_30 = past_early[past_early['days_from_start'] <= 30]
early_features = early_30.groupby('user_id').agg(
    revenue_first_30d=('total_price', 'sum'),
    orders_first_30d=('total_price', 'count'),
    quantity_first_30d=('quantity', 'sum'),
    categories_first_30d=('category', 'nunique')
).reset_index()

early_features['avg_order_value_first_30d'] = (
    early_features['revenue_first_30d'] / early_features['orders_first_30d'].replace(0, np.nan)
)

user = user.merge(early_features, on='user_id', how='left')
user.head()

### 5.3 Доля топ-категории

Этот признак отражает специализацию покупок. Если большая часть заказов приходится на одну категорию, пользователь имеет более устойчивый категорийный паттерн.

In [ ]:
cat_counts = past.groupby(['user_id', 'category']).size().rename('cat_orders').reset_index()
top_cat = cat_counts.loc[cat_counts.groupby('user_id')['cat_orders'].idxmax()].copy()
total_orders_by_user = past.groupby('user_id').size().rename('num_orders_check').reset_index()

top_cat = top_cat.merge(total_orders_by_user, on='user_id', how='left')
top_cat['top_category_share'] = top_cat['cat_orders'] / top_cat['num_orders_check']
top_cat = top_cat[['user_id', 'category', 'top_category_share']].rename(columns={'category': 'top_category'})

user = user.merge(top_cat, on='user_id', how='left')
user.head()

### 5.4 Тренды активности

Сравниваем последние 90 дней исторического периода с предыдущими 90 днями. Это позволяет увидеть рост или падение активности до даты отсечения.

In [ ]:
last_90_start = cutoff - pd.Timedelta(days=90)
prev_90_start = cutoff - pd.Timedelta(days=180)

last_90 = past[(past['order_date'] >= last_90_start) & (past['order_date'] < cutoff)]
prev_90 = past[(past['order_date'] >= prev_90_start) & (past['order_date'] < last_90_start)]

last_90_features = last_90.groupby('user_id').agg(
    revenue_last_90d=('total_price', 'sum'),
    orders_last_90d=('total_price', 'count')
).reset_index()

prev_90_features = prev_90.groupby('user_id').agg(
    revenue_prev_90d=('total_price', 'sum'),
    orders_prev_90d=('total_price', 'count')
).reset_index()

user = user.merge(last_90_features, on='user_id', how='left')
user = user.merge(prev_90_features, on='user_id', how='left')

for col in ['revenue_last_90d', 'orders_last_90d', 'revenue_prev_90d', 'orders_prev_90d']:
    user[col] = user[col].fillna(0)

user['revenue_trend_90d'] = (user['revenue_last_90d'] + 1) / (user['revenue_prev_90d'] + 1)
user['orders_trend_90d'] = (user['orders_last_90d'] + 1) / (user['orders_prev_90d'] + 1)

user.head()

### 5.5 Бинарные поведенческие флаги

In [ ]:
user['is_repeat_buyer'] = (user['num_orders'] > 1).astype(int)
user['is_recent_buyer_90d'] = (user['recency'] <= 90).astype(int)
user['has_many_categories'] = (user['category_count'] > user['category_count'].median()).astype(int)

user.head()

## 6. Итоговый датасет для моделирования

In [ ]:
data = user.merge(ltv, on='user_id', how='left')
data['ltv'] = data['ltv'].fillna(0)
data['ltv_log'] = np.log1p(data['ltv'])

# Удаляем даты из модели, но оставляем рассчитанные на их основе числовые признаки
date_cols = ['first_order_date', 'last_order_date']

# Заполняем пропуски, которые появились у пользователей с одной покупкой
num_cols = data.select_dtypes(include=[np.number]).columns.tolist()
for col in num_cols:
    if col not in ['ltv', 'ltv_log']:
        data[col] = data[col].replace([np.inf, -np.inf], np.nan)
        data[col] = data[col].fillna(0)

# Категориальные признаки
for col in ['top_category']:
    data[col] = data[col].fillna('unknown')

data.shape

In [ ]:
data.head()

In [ ]:
data[['ltv', 'ltv_log']].describe(percentiles=[.01, .05, .25, .5, .75, .95, .99]).T

## 7. EDA пользовательского уровня

In [ ]:
plt.figure(figsize=(10, 4))
sns.histplot(data['ltv'], bins=60)
plt.title('Распределение LTV')
plt.show()

plt.figure(figsize=(10, 4))
sns.histplot(data['ltv_log'], bins=60)
plt.title('Распределение log1p(LTV)')
plt.show()

In [ ]:
important_features = [
    'total_revenue', 'num_orders', 'avg_order_value', 'median_order_value', 'std_order_value',
    'recency', 'customer_age', 'tenure', 'purchase_frequency_rate', 'revenue_per_day',
    'avg_days_between_orders', 'std_days_between_orders', 'category_count', 'product_count',
    'top_category_share', 'revenue_first_30d', 'orders_first_30d',
    'revenue_last_90d', 'orders_last_90d', 'revenue_trend_90d', 'orders_trend_90d'
]

available_important_features = [c for c in important_features if c in data.columns]

data[available_important_features + ['ltv', 'ltv_log']].describe(percentiles=[.01, .05, .25, .5, .75, .95, .99]).T

### 7.1 Корреляции с LTV

Используем Pearson и Spearman. Pearson показывает линейную связь, Spearman — монотонную связь и более устойчив к выбросам.

In [ ]:
numeric_features = [c for c in data.select_dtypes(include=[np.number]).columns if c not in ['ltv', 'ltv_log']]

pearson_corr = data[numeric_features + ['ltv_log']].corr(method='pearson')['ltv_log'].sort_values(ascending=False)
spearman_corr = data[numeric_features + ['ltv_log']].corr(method='spearman')['ltv_log'].sort_values(ascending=False)

corr_table = pd.DataFrame({
    'pearson_corr_with_ltv_log': pearson_corr,
    'spearman_corr_with_ltv_log': spearman_corr
}).drop(index='ltv_log')

corr_table.sort_values('spearman_corr_with_ltv_log', ascending=False).head(30)

In [ ]:
top_corr_features = corr_table['spearman_corr_with_ltv_log'].abs().sort_values(ascending=False).head(20).index.tolist()

plt.figure(figsize=(12, 10))
sns.heatmap(data[top_corr_features + ['ltv_log']].corr(method='spearman'), cmap='coolwarm', center=0, annot=False)
plt.title('Spearman correlation heatmap: топ-признаки и LTV')
plt.show()

### 7.2 Проверка мультиколлинеарности

Если признаки сильно коррелируют между собой, линейные модели могут быть нестабильны. Для деревьев это менее критично, но всё равно полезно понимать дублирование признаков.

In [ ]:
corr_matrix = data[numeric_features].corr(method='spearman').abs()
upper = corr_matrix.where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))
high_corr_pairs = (
    upper.stack()
    .reset_index()
    .rename(columns={'level_0': 'feature_1', 'level_1': 'feature_2', 0: 'abs_spearman_corr'})
    .sort_values('abs_spearman_corr', ascending=False)
)
high_corr_pairs.head(25)

### 7.3 Визуализация зависимостей ключевых признаков с LTV

In [ ]:
plot_features = top_corr_features[:8]

for col in plot_features:
    plt.figure(figsize=(7, 4))
    sns.scatterplot(x=np.log1p(data[col]), y=data['ltv_log'], alpha=0.5)
    sns.regplot(x=np.log1p(data[col]), y=data['ltv_log'], scatter=False, lowess=True)
    plt.title(f'Зависимость log1p(LTV) от log1p({col})')
    plt.xlabel(f'log1p({col})')
    plt.ylabel('log1p(LTV)')
    plt.show()

### 7.4 Анализ категорий

In [ ]:
top_categories = data['top_category'].value_counts().head(20)
top_categories

In [ ]:
category_ltv = (
    data.groupby('top_category')
    .agg(users=('user_id', 'count'), median_ltv=('ltv', 'median'), mean_ltv=('ltv', 'mean'), median_ltv_log=('ltv_log', 'median'))
    .query('users >= 20')
    .sort_values('median_ltv_log', ascending=False)
)
category_ltv.head(20)

## 8. Подготовка к моделированию

In [ ]:
target = 'ltv_log'
exclude_cols = ['user_id', 'ltv', 'ltv_log'] + date_cols

features = [c for c in data.columns if c not in exclude_cols]
X = data[features].copy()
y = data[target].copy()

numeric_features = X.select_dtypes(include=[np.number]).columns.tolist()
categorical_features = X.select_dtypes(exclude=[np.number]).columns.tolist()

numeric_features, categorical_features

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42
)

preprocess_linear = ColumnTransformer(
    transformers=[
        ('num', Pipeline(steps=[
            ('imputer', SimpleImputer(strategy='median')),
            ('scaler', StandardScaler())
        ]), numeric_features),
        ('cat', Pipeline(steps=[
            ('imputer', SimpleImputer(strategy='most_frequent')),
            ('onehot', OneHotEncoder(handle_unknown='ignore', min_frequency=20))
        ]), categorical_features)
    ]
)

preprocess_tree = ColumnTransformer(
    transformers=[
        ('num', SimpleImputer(strategy='median'), numeric_features),
        ('cat', Pipeline(steps=[
            ('imputer', SimpleImputer(strategy='most_frequent')),
            ('onehot', OneHotEncoder(handle_unknown='ignore', min_frequency=20))
        ]), categorical_features)
    ]
)

## 9. Обучение и сравнение моделей

Модели сравниваются на `ltv_log`, потому что исходный LTV имеет сильную правостороннюю асимметрию. Дополнительно метрики переводятся обратно в денежную шкалу через `expm1`.

In [ ]:
def evaluate_model(model, X_train, X_test, y_train, y_test, model_name):
    model.fit(X_train, y_train)
    pred_log = model.predict(X_test)
    
    mae_log = mean_absolute_error(y_test, pred_log)
    rmse_log = mean_squared_error(y_test, pred_log, squared=False)
    r2 = r2_score(y_test, pred_log)
    
    y_true = np.expm1(y_test)
    y_pred = np.expm1(pred_log)
    
    mae = mean_absolute_error(y_true, y_pred)
    rmse = mean_squared_error(y_true, y_pred, squared=False)
    
    return {
        'model': model_name,
        'MAE_log': mae_log,
        'RMSE_log': rmse_log,
        'R2_log': r2,
        'MAE_original_scale': mae,
        'RMSE_original_scale': rmse
    }, model

models = {
    'LinearRegression': Pipeline([('prep', preprocess_linear), ('model', LinearRegression())]),
    'Ridge': Pipeline([('prep', preprocess_linear), ('model', Ridge(alpha=1.0))]),
    'Lasso': Pipeline([('prep', preprocess_linear), ('model', Lasso(alpha=0.001, max_iter=5000))]),
    'RandomForest': Pipeline([('prep', preprocess_tree), ('model', RandomForestRegressor(
        n_estimators=300, max_depth=8, min_samples_leaf=5, random_state=42, n_jobs=-1
    ))]),
    'HistGradientBoosting': Pipeline([('prep', preprocess_tree), ('model', HistGradientBoostingRegressor(
        max_iter=300, learning_rate=0.05, max_leaf_nodes=31, random_state=42
    ))])
}

results = []
fitted_models = {}
for name, model in models.items():
    res, fitted = evaluate_model(model, X_train, X_test, y_train, y_test, name)
    results.append(res)
    fitted_models[name] = fitted

results_df = pd.DataFrame(results).sort_values('RMSE_log')
results_df

## 10. Интерпретация модели

Для дерева/бустинга можно использовать permutation importance. Этот метод показывает, насколько ухудшается качество модели, если значения признака случайно перемешать.

In [ ]:
best_model_name = results_df.iloc[0]['model']
best_model = fitted_models[best_model_name]
best_model_name

In [ ]:
perm = permutation_importance(
    best_model, X_test, y_test,
    n_repeats=10,
    random_state=42,
    scoring='neg_root_mean_squared_error',
    n_jobs=-1
)

perm_df = pd.DataFrame({
    'feature': X_test.columns,
    'importance_mean': perm.importances_mean,
    'importance_std': perm.importances_std
}).sort_values('importance_mean', ascending=False)

perm_df.head(25)

In [ ]:
plt.figure(figsize=(10, 7))
sns.barplot(data=perm_df.head(20), x='importance_mean', y='feature')
plt.title(f'Permutation importance для модели {best_model_name}')
plt.xlabel('Среднее падение качества при перемешивании признака')
plt.ylabel('Признак')
plt.show()

## 11. Проверка ошибок модели

Важно не только посмотреть R²/RMSE, но и понять, где модель ошибается: на низких, средних или высоких значениях LTV.

In [ ]:
pred_log = best_model.predict(X_test)
errors = pd.DataFrame({
    'y_true_log': y_test,
    'y_pred_log': pred_log,
    'y_true': np.expm1(y_test),
    'y_pred': np.expm1(pred_log)
})
errors['abs_error'] = np.abs(errors['y_true'] - errors['y_pred'])
errors['residual_log'] = errors['y_true_log'] - errors['y_pred_log']

errors.describe(percentiles=[.01, .05, .25, .5, .75, .95, .99]).T

In [ ]:
plt.figure(figsize=(7, 5))
sns.scatterplot(data=errors, x='y_true_log', y='y_pred_log', alpha=0.6)
plt.plot([errors['y_true_log'].min(), errors['y_true_log'].max()], [errors['y_true_log'].min(), errors['y_true_log'].max()], linestyle='--')
plt.title('Фактический vs прогнозный log1p(LTV)')
plt.xlabel('Фактический log1p(LTV)')
plt.ylabel('Прогнозный log1p(LTV)')
plt.show()

plt.figure(figsize=(8, 4))
sns.histplot(errors['residual_log'], bins=40)
plt.title('Распределение ошибок на лог-шкале')
plt.xlabel('Фактическое - прогнозное')
plt.show()

## 12. Выводы для диплома

После выполнения блокнота этот раздел нужно заполнить фактическими результатами:

1. какие признаки имеют наибольшую корреляцию с `ltv_log`;
2. какие признаки оказались важными по permutation importance;
3. какая модель показала лучшую метрику RMSE/R²;
4. какие поведенческие паттерны можно считать значимыми.

Ожидаемые группы значимых факторов:
- историческая денежная активность: `total_revenue`, `revenue_per_day`, `avg_order_value`;
- частота покупок: `num_orders`, `purchase_frequency_rate`, `orders_per_active_day`;
- регулярность: `avg_days_between_orders`, `std_days_between_orders`;
- раннее поведение: `revenue_first_30d`, `orders_first_30d`;
- категориальное разнообразие: `category_count`, `top_category_share`, `category_entropy`;
- недавняя активность: `recency`, `orders_last_90d`, `revenue_last_90d`.
